# Roketsan vehicle detection: four-detector ensemble

Four detectors are trained separately, and their test predictions are fused box by box:

| Model | Input | Trained on | Hardware | Schedule | Weights used | Fusion weight |
|---|---|---|---|---|---|---|
| YOLO11m, repeat-factor sampling | full image, 1280 px | scene_holdout_v2 train | 2x T4 | 30 epochs | `last.pt` | 0.6 |
| YOLO11m | full image, 1280 px | scene_holdout_v2 train | 2x T4 | up to 100 epochs, patience 20 | `best.pt` | 0.8 |
| YOLO26-S-P2 | 704 px tiles | v2 train, then all labelled images | 1x A100 | up to 40 epochs + 3 epoch fine-tune | `last.pt` | 0.4 |
| RF-DETR Large + flip TTA | 704 px tiles | v2 train, then all labelled images | 1x A100 | 12 epochs + 3 epoch fine-tune | last EMA | 1.0 |

**Validation.** Every model is first trained on the 5,176 scene_holdout_v2 training images and checked on its 1,295
validation images. The split keeps whole scenes on one side, and the validation images are chosen to match the test
set's resolution and brightness mix. The fusion weights were picked on those validation predictions. The two tiled
models then get a short fine-tune on all 6,471 labelled images before predicting on test.

The training sections (3-7) don't depend on each other, so each can run on a different machine. Sections 8 and 9
need all four weight files.

## 1. Setup

In [ ]:
# pip install ultralytics==8.4.163 \
#     "rfdetr @ git+https://github.com/roboflow/rf-detr@4f68814adb3df5a1ac4769fc11f7b32d8e2d404f"

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")  # less fragmentation at 1280 px

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision.ops import batched_nms, box_iou
from ultralytics import YOLO

DATA = Path("data")      # train/annotations.csv, train/images, test/images, sample_submission.csv
SPLITS = Path("splits")  # scene_holdout_v2/train.txt, val.txt
WORK = Path("work")      # generated datasets, training runs, predictions
WORK.mkdir(exist_ok=True)

TRAIN_IMAGES, TEST_IMAGES = DATA / "train/images", DATA / "test/images"
CLASSES = ["car", "van", "truck", "bus"]
SEED = 42

SH2 = {k: (SPLITS / "scene_holdout_v2" / f"{k}.txt").read_text().split() for k in ("train", "val")}
assert len(SH2["train"]) == 5176 and len(SH2["val"]) == 1295 and not set(SH2["train"]) & set(SH2["val"])
ALL_LABELLED = SH2["train"] + SH2["val"]


def image_sizes(folder, ids):
    sizes = {}
    for i in ids:
        with Image.open(folder / f"{i}.jpg") as im:
            sizes[i] = im.size  # (width, height)
    return sizes


ann = pd.read_csv(DATA / "train/annotations.csv")
ann["aid"] = np.arange(1, len(ann) + 1)  # stable annotation id (CSV row), seeds the crop jitter below
train_sizes = image_sizes(TRAIN_IMAGES, ALL_LABELLED)
print(f"{len(ann):,} boxes on {ann.image_id.nunique():,} images")
print(ann.label.value_counts().reindex(CLASSES).to_string())

## 2. Full-image YOLO dataset (both YOLO11m models)

Both YOLO11m models train on whole images. Boxes are clipped to the image and written in YOLO format (class, cx,
cy, w, h, normalised).

In [ ]:
YOLO_FULL = WORK / "yolo_full"


def yolo_label_lines(width, height, boxes):
    lines = []
    for b in boxes.itertuples():
        x1, y1 = max(b.x, 0), max(b.y, 0)
        x2, y2 = min(b.x + b.w, width), min(b.y + b.h, height)
        if x2 > x1 and y2 > y1:
            lines.append(f"{CLASSES.index(b.label)} {(x1 + x2) / 2 / width:.6f} {(y1 + y2) / 2 / height:.6f} "
                         f"{(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}")
    return lines


def link_or_copy(src, dst):
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


def write_data_yaml(path, root, train, val):
    names = "".join(f"  {k}: {c}\n" for k, c in enumerate(CLASSES))
    path.write_text(f"path: {root.resolve().as_posix()}\ntrain: {train}\nval: {val}\nnames:\n{names}")
    return path


boxes_by_image = dict(tuple(ann.groupby("image_id")))
for split, ids in SH2.items():
    (YOLO_FULL / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_FULL / "labels" / split).mkdir(parents=True, exist_ok=True)
    for i in ids:
        dst = YOLO_FULL / "images" / split / f"{i}.jpg"
        if not dst.exists():
            link_or_copy(TRAIN_IMAGES / f"{i}.jpg", dst)
        lines = yolo_label_lines(*train_sizes[i], boxes_by_image.get(i, ann.iloc[:0]))
        (YOLO_FULL / "labels" / split / f"{i}.txt").write_text("".join(f"{l}\n" for l in lines))

## 3. YOLO11m with repeat-factor sampling

Buses (3% of boxes) and trucks (7%) are rare, but each class counts for a quarter of mAP. Instead of a class-weighted
loss, rare classes are oversampled by listing their images several times in the training list (Ultralytics keeps
duplicates):

- image with a bus: x3
- image with a truck but no bus: x2
- everything else: x1

That gives 9,986 entries per epoch instead of 5,176 (1.93x), so `warmup_epochs` and `close_mosaic` are divided by
1.93 to keep the same number of iterations as an unweighted run. Test predictions use `last.pt`, because `best.pt`
is selected on the validation set we score on.

In [ ]:
def repeat_factor(image_id):
    labels = set(boxes_by_image[image_id].label) if image_id in boxes_by_image else set()
    return 3 if "bus" in labels else 2 if "truck" in labels else 1


rfs_list = [f"./images/train/{i}.jpg\n" for i in SH2["train"] for _ in range(repeat_factor(i))]
(YOLO_FULL / "train_rfs.txt").write_text("".join(rfs_list))
print(f"{len(rfs_list):,} training entries from {len(SH2['train']):,} images")

rfs_yaml = write_data_yaml(WORK / "yolo11m_rfs.yaml", YOLO_FULL, "train_rfs.txt", "images/val")

In [ ]:
YOLO("yolo11m.pt").train(
    data=str(rfs_yaml),
    imgsz=1280,
    epochs=30,
    batch=8,             # 4 per T4; 8 per T4 runs out of memory at 1280
    device=[0, 1],
    workers=4,
    fliplr=0.5,
    flipud=0.5,          # aerial views: vertical flips are as valid as horizontal ones
    degrees=0.0,
    scale=0.4,
    mosaic=1.0,
    close_mosaic=5,      # 10 / 1.93
    warmup_epochs=1.55,  # 3 / 1.93
    cos_lr=True,
    patience=10,
    amp=True,
    save_period=5,
    seed=0,
    project=str(WORK / "runs"),
    name="yolo11m_rfs",
    exist_ok=True,
    plots=False,
)
WEIGHTS_RFS = WORK / "runs/yolo11m_rfs/weights/last.pt"

## 4. YOLO11m, 100 epochs

Same architecture and resolution, with Ultralytics' default augmentation and no resampling. Training runs for up to
100 epochs and stops once validation mAP50-95 hasn't improved for 20 epochs. `nbs=64` (the default) accumulates
gradients up to 64 images, so the per-GPU batch of 4 only affects memory.

In [ ]:
base_yaml = write_data_yaml(WORK / "yolo11m_base.yaml", YOLO_FULL, "images/train", "images/val")

YOLO("yolo11m.pt").train(
    data=str(base_yaml),
    imgsz=1280,
    epochs=100,
    patience=20,
    batch=8,  # 4 per T4
    device=[0, 1],
    workers=4,
    seed=SEED,
    cache=False,
    project=str(WORK / "runs"),
    name="yolo11m_base",
    exist_ok=True,
)
WEIGHTS_BASE = WORK / "runs/yolo11m_base/weights/best.pt"

## 5. Tiled datasets (YOLO26 and RF-DETR)

Most vehicles are under 32 px, so the other two models train on 704 px crops at native resolution instead of
downscaled whole images. Both use the same tile recipe:

- **Grid tiles:** 704 px, 25% overlap. A box belongs to a tile if at least 60% of its area is inside.
- **Negatives:** every tile with a box is kept; empty tiles are subsampled to 20% of that count.
- **Rare-class crops:** extra tiles centred on a van, truck or bus, 25% as many as the positive grid tiles, split
  55% van / 30% truck / 15% bus. Vans with a car in the same crop come first, because car vs van is the hardest
  distinction in the data. The centre is jittered by up to 15% of the tile, the anchor box must be 80% visible, and
  an image gets at most 4 such crops.

All the random choices are hashes of fixed keys, so the tile set is identical on every run.

In [ ]:
TILE, OVERLAP, MIN_VISIBLE = 704, 0.25, 0.60
FOCUS_SHARES = {"van": 0.55, "truck": 0.30, "bus": 0.15}


def stable_hex(*parts):
    return hashlib.sha256(":".join(map(str, parts)).encode()).hexdigest()


def stable_unit(*parts):
    return int(stable_hex(*parts)[:16], 16) / float(16**16 - 1)


def tile_origins(length):
    if length <= TILE:
        return [0]
    step = round(TILE * (1 - OVERLAP))
    origins = list(range(0, length - TILE + 1, step))
    if origins[-1] != length - TILE:
        origins.append(length - TILE)  # last tile flush with the edge
    return origins


def clip_to_tile(b, left, top, min_visible=MIN_VISIBLE):
    x1, y1 = max(b.x, left), max(b.y, top)
    x2, y2 = min(b.x + b.w, left + TILE), min(b.y + b.h, top + TILE)
    if x2 <= x1 or y2 <= y1 or (x2 - x1) * (y2 - y1) / max(1.0, b.w * b.h) < min_visible:
        return None
    return b.label, x1 - left, y1 - top, x2 - x1, y2 - y1


def make_tile(image_id, left, top, boxes, kind="grid", focus=None, aid=0):
    kept = [c for b in boxes for c in [clip_to_tile(b, left, top)] if c]
    return {"image_id": image_id, "left": left, "top": top, "kind": kind, "focus": focus, "aid": aid, "boxes": kept}


def grid_tiles(ids, boxes):
    return [make_tile(i, left, top, boxes.get(i, []))
            for i in ids
            for top in tile_origins(train_sizes[i][1])
            for left in tile_origins(train_sizes[i][0])]


def subsample_negatives(tiles, ratio=0.20):
    pos = [t for t in tiles if t["boxes"]]
    neg = sorted((t for t in tiles if not t["boxes"]),
                 key=lambda t: stable_hex(SEED, t["image_id"], t["left"], t["top"], "negative"))
    return pos + neg[:round(len(pos) * ratio)]


def crop_origin(b, image_id, jitter, attempt):
    w, h = train_sizes[image_id]
    dx = (stable_unit(SEED, b.aid, attempt, "x") * 2 - 1) * jitter * TILE
    dy = (stable_unit(SEED, b.aid, attempt, "y") * 2 - 1) * jitter * TILE
    left = min(max(0.0, b.x + b.w / 2 + dx - TILE / 2), max(0, w - TILE))
    top = min(max(0.0, b.y + b.h / 2 + dy - TILE / 2), max(0, h - TILE))
    return round(left), round(top)


def class_quotas(total):
    raw = {c: total * s for c, s in FOCUS_SHARES.items()}
    quotas = {c: math.floor(v) for c, v in raw.items()}
    by_remainder = sorted(FOCUS_SHARES, key=lambda c: (-(raw[c] - quotas[c]), list(FOCUS_SHARES).index(c)))
    for c in by_remainder[:total - sum(quotas.values())]:
        quotas[c] += 1
    return quotas


def centred_tiles(ids, boxes, grid, ratio=0.25, jitter=0.15, anchor_visible=0.80, max_per_image=4):
    quotas = class_quotas(round(sum(bool(t["boxes"]) for t in grid) * ratio))
    used = {(t["image_id"], t["left"], t["top"]) for t in grid}
    per_image, out = {}, []
    for cls, quota in quotas.items():
        candidates = []
        for i in ids:
            for b in boxes.get(i, []):
                if b.label != cls:
                    continue
                near_car = cls == "van" and any(
                    o.label == "car" and clip_to_tile(o, *crop_origin(b, i, 0.0, 0)) for o in boxes[i])
                candidates.append((not near_car, stable_hex(SEED, cls, i, b.aid), i, b))
        candidates.sort(key=lambda c: c[:2])

        accepted = 0
        for _, _, i, b in candidates:
            if accepted == quota:
                break
            if per_image.get(i, 0) >= max_per_image:
                continue
            for attempt in range(10):
                left, top = crop_origin(b, i, jitter, attempt)
                if (i, left, top) not in used and clip_to_tile(b, left, top, anchor_visible):
                    break
            else:
                continue
            out.append(make_tile(i, left, top, boxes[i], "centred", cls, b.aid))
            used.add((i, left, top))
            per_image[i] = per_image.get(i, 0) + 1
            accepted += 1
        assert accepted == quota, f"only {accepted}/{quota} {cls} crops"
    return out


def training_tiles(ids, boxes):
    grid = grid_tiles(ids, boxes)
    return subsample_negatives(grid) + centred_tiles(ids, boxes, grid)


def crops(tiles):
    """(tile, file name, PIL crop), opening each source image once."""
    current = image = None
    for t in sorted(tiles, key=lambda t: (t["image_id"], t["kind"], t["top"], t["left"], t["aid"])):
        if t["image_id"] != current:
            current = t["image_id"]
            image = Image.open(TRAIN_IMAGES / f"{current}.jpg").convert("RGB")
        suffix = f"focus-{t['focus']}-a{t['aid']}" if t["kind"] == "centred" else "grid"
        # crop() pads past the image edge with black, so every tile is TILE x TILE
        yield t, f"{current}__{suffix}_x{t['left']}_y{t['top']}.jpg", image.crop(
            (t["left"], t["top"], t["left"] + TILE, t["top"] + TILE))


def boxes_for(frame):
    return {i: list(g.itertuples()) for i, g in frame.groupby("image_id")}

## 6. YOLO26-S-P2 on tiles

YOLO26-S with an extra P2 head (stride 4) for small objects. Duplicate annotation rows are dropped before tiling.

**Stage 1, scene_holdout_v2 train tiles.** AdamW at `lr0=5e-4`, which beat `1e-3` in a short pilot. Augmentation is
lighter than in the YOLO11m runs: mosaic 0.3, small rotations, no vertical flip. Validation runs every epoch on the
full-size val images, and training stops after 6 epochs without improvement. In the original run the first 9 epochs
were the pilot itself (no validation) and training resumed from there. It stopped after epoch 38; the best
checkpoint was epoch 31.

**Stage 2, all labelled images.** 3 more epochs from that `best.pt` with every other setting carried over. The
learning rate starts at `1e-4` and decays to `1e-5`, with no warm-up and mosaic off throughout. The val images are
now in the training set, so validation is only a sanity check and the final `last.pt` is used.

In [ ]:
dedup = ann.drop_duplicates(["image_id", "x", "y", "w", "h", "label"])


def write_yolo_tiles(root, train_ids):
    (root / "images/train").mkdir(parents=True, exist_ok=True)
    (root / "labels/train").mkdir(parents=True, exist_ok=True)
    tiles = training_tiles(train_ids, boxes_for(dedup))
    for t, name, crop in crops(tiles):
        crop.save(root / "images/train" / name, quality=92)
        lines = [f"{CLASSES.index(c)} {(x + w / 2) / TILE:.8f} {(y + h / 2) / TILE:.8f} {w / TILE:.8f} {h / TILE:.8f}"
                 for c, x, y, w, h in t["boxes"]]
        (root / "labels/train" / name).with_suffix(".txt").write_text("".join(f"{l}\n" for l in lines))
    # validation: the full-size val images with the labels from section 2
    for sub in ("images", "labels"):
        shutil.rmtree(root / sub / "val", ignore_errors=True)
        shutil.copytree(YOLO_FULL / sub / "val", root / sub / "val", symlinks=True)
    print(f"{root.name}: {len(tiles):,} training tiles")
    return write_data_yaml(root / "data.yaml", root, "images/train", "images/val")


YOLO26_ARGS = dict(
    imgsz=TILE,
    batch=0.80,  # auto batch size: 80% of GPU memory
    optimizer="AdamW",
    weight_decay=5e-4,
    cos_lr=True,
    amp="bf16",
    seed=SEED,
    deterministic=True,
    mosaic=0.30,
    mixup=0.0,
    translate=0.08,
    scale=0.30,
    fliplr=0.5,
    flipud=0.0,
    degrees=3.0,
    device=0,
    workers=12,
    cache=False,
    val=True,
    plots=False,
    save_period=1,
    project=str(WORK / "runs"),
)

**Stage 1: scene_holdout_v2, up to 40 epochs**

In [ ]:
v2_tiles_yaml = write_yolo_tiles(WORK / "yolo_tiles_v2", SH2["train"])

YOLO("yolo26s-p2.yaml").load("yolo26s.pt").train(
    data=str(v2_tiles_yaml), epochs=40, patience=6, lr0=5e-4, lrf=0.01, warmup_epochs=1.0, close_mosaic=5,
    name="yolo26_v2", **YOLO26_ARGS,
)

**Stage 2: fine-tune on all labelled images, 3 epochs**

In [ ]:
all_tiles_yaml = write_yolo_tiles(WORK / "yolo_tiles_all", ALL_LABELLED)

YOLO(str(WORK / "runs/yolo26_v2/weights/best.pt")).train(
    data=str(all_tiles_yaml), epochs=3, patience=4, lr0=1e-4, lrf=0.1, warmup_epochs=0,
    close_mosaic=3,  # mosaic off for every epoch, like the end of stage 1
    name="yolo26_all", **YOLO26_ARGS,
)
WEIGHTS_YOLO26 = WORK / "runs/yolo26_all/weights/last.pt"

## 7. RF-DETR Large on tiles

RF-DETR is a transformer detector (DETR family, DINOv2 backbone). It's a different design from the three YOLOs,
which is what makes it useful in the ensemble. Tiles are built from the full annotation file, without the
deduplication used for YOLO26.

**Stage 1, scene_holdout_v2 train tiles.** 12 epochs from COCO weights at `lr=1e-4` (backbone `1e-5`), bf16, EMA
weights, evaluated every 3 epochs. The best EMA checkpoint (epoch 12) goes on to stage 2.

**Stage 2, all labelled images.** 3 epochs from that checkpoint at `3e-5` / `3e-6`. The final EMA weights go to
test.

The original runs used an RF-DETR build patched for per-class loss weights. All the weights were 1.0, which leaves
the loss unchanged, so the stock `rfdetr` package at the same commit reproduces them.

In [ ]:
from rfdetr import RFDETRLarge

COCO_CATEGORIES = [{"id": k, "name": c, "supercategory": "vehicle"} for k, c in enumerate(CLASSES)]


def write_coco_tiles(tiles, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    images, annotations = [], []
    for n, (t, name, crop) in enumerate(crops(tiles), start=1):
        crop.save(out_dir / name, quality=92)
        images.append({"id": n, "file_name": name, "width": TILE, "height": TILE})
        for c, x, y, w, h in t["boxes"]:
            annotations.append({"id": len(annotations) + 1, "image_id": n, "category_id": CLASSES.index(c),
                                "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
    coco = {"images": images, "annotations": annotations, "categories": COCO_CATEGORIES}
    (out_dir / "_annotations.coco.json").write_text(json.dumps(coco))


def rfdetr_dataset(root, train_ids):
    boxes = boxes_for(ann)
    write_coco_tiles(training_tiles(train_ids, boxes), root / "train")
    write_coco_tiles(grid_tiles(SH2["val"], boxes), root / "valid")  # every grid tile of the val images
    return root


rfdetr_common = dict(batch_size=19, grad_accum_steps=1, weight_decay=1e-4, resolution=TILE, amp_dtype="bf16",
                     use_ema=True, checkpoint_interval=1, eval_max_dets=500, early_stopping=False, seed=SEED,
                     num_workers=12)

**Stage 1: scene_holdout_v2, 12 epochs**

In [ ]:
stage1_out = WORK / "runs/rfdetr_v2"
RFDETRLarge(resolution=TILE).train(
    dataset_dir=str(rfdetr_dataset(WORK / "rfdetr_v2", SH2["train"])), output_dir=str(stage1_out),
    epochs=12, lr=1e-4, lr_encoder=1e-5, eval_interval=3, **rfdetr_common,
)

**Stage 2: fine-tune on all labelled images, 3 epochs**

The v2 val tiles are still passed as `valid` because RF-DETR expects a validation set. They're part of the training
data here, so the validation score means nothing.

In [ ]:
stage2_out = WORK / "runs/rfdetr_all"
RFDETRLarge(resolution=TILE, pretrain_weights=str(stage1_out / "checkpoint_best_ema.pth"),
            num_classes=len(CLASSES)).train(
    dataset_dir=str(rfdetr_dataset(WORK / "rfdetr_all", ALL_LABELLED)), output_dir=str(stage2_out),
    epochs=3, lr=3e-5, lr_encoder=3e-6, eval_interval=3, **rfdetr_common,
)
WEIGHTS_RFDETR = stage2_out / "last_ema.pth"

## 8. Test predictions

Every model predicts at `conf=0.001`, because the fusion and mAP both need the low-confidence tail.

- **YOLO11m (both):** whole image at 1280 px, NMS IoU 0.7. Up to 1000 boxes for the repeat-factor model, 300 for
  the other (each model's own setting).
- **YOLO26 and RF-DETR:** each test image is cut into the same 704 px grid as in training. A box is kept only by
  the tile that owns its centre (neighbouring tiles split their overlap down the middle), so objects cut by a tile
  edge aren't counted twice. Boxes under 200 px² are dropped, class-wise NMS at 0.6 removes the remaining
  duplicates, and at most 500 boxes are kept per image.
- **RF-DETR flip TTA:** every tile is also predicted mirrored left-right, and those boxes are mirrored back. After
  NMS within each view, each box is paired with its best same-class match from the other view (IoU >= 0.55, one to
  one). A matched pair becomes one box, with coordinates weighted by score and the mean of the two scores. A box
  with no partner keeps half its score.

In [ ]:
PRED_COLS = ["image_id", "label", "conf", "x", "y", "w", "h"]
sample = pd.read_csv(DATA / "sample_submission.csv")
TEST_IDS = sample.image_id.tolist()
test_sizes = image_sizes(TEST_IMAGES, TEST_IDS)
(WORK / "preds").mkdir(exist_ok=True)


def to_frame(image_id, xyxy, scores, labels):
    return pd.DataFrame({"image_id": image_id, "label": [CLASSES[int(c)] for c in labels], "conf": scores,
                         "x": xyxy[:, 0], "y": xyxy[:, 1],
                         "w": xyxy[:, 2] - xyxy[:, 0], "h": xyxy[:, 3] - xyxy[:, 1]})


def predict_full(weights, max_det):
    model, parts = YOLO(str(weights)), []
    for i in TEST_IDS:
        r = model.predict(str(TEST_IMAGES / f"{i}.jpg"), imgsz=1280, conf=0.001, iou=0.7, max_det=max_det,
                          half=True, verbose=False)[0]
        parts.append(to_frame(i, r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()))
    df = pd.concat(parts, ignore_index=True)
    return df[(df.w >= 1) & (df.h >= 1)]  # zero-size boxes clipped at the edge aren't valid in a submission


def owned_span(origins, k, length):
    centres = [o + TILE / 2 for o in origins]
    lo = 0.0 if k == 0 else (centres[k - 1] + centres[k]) / 2
    hi = float(length) if k == len(centres) - 1 else (centres[k] + centres[k + 1]) / 2
    return lo, hi


def nms(xyxy, s, c, iou=0.6):
    keep = batched_nms(torch.from_numpy(xyxy).float(), torch.from_numpy(s).float(),
                       torch.from_numpy(c.astype(np.int64)), iou).numpy()
    return xyxy[keep], s[keep], c[keep]


def fuse_flip_views(a, b, iou_thr=0.55):
    """One-to-one fusion of the plain view a and flipped view b (each score-sorted after NMS)."""
    out_xyxy, out_s, out_c = [], [], []
    for cls in np.union1d(a[2], b[2]):
        (xa, sa), (xb, sb) = [(v[0][v[2] == cls], v[1][v[2] == cls]) for v in (a, b)]
        iou = box_iou(torch.from_numpy(xa), torch.from_numpy(xb)).numpy()
        taken = np.zeros(len(xb), bool)
        for i in range(len(xa)):
            row = np.where(taken, -1.0, iou[i])
            j = int(row.argmax()) if len(row) else -1
            if j >= 0 and row[j] >= iou_thr:
                taken[j] = True
                out_xyxy.append((xa[i] * sa[i] + xb[j] * sb[j]) / (sa[i] + sb[j]))
                out_s.append((sa[i] + sb[j]) / 2)
            else:
                out_xyxy.append(xa[i])
                out_s.append(sa[i] / 2)
            out_c.append(cls)
        for j in np.flatnonzero(~taken):
            out_xyxy.append(xb[j])
            out_s.append(sb[j] / 2)
            out_c.append(cls)
    if not out_s:
        return np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, np.int64)
    return np.array(out_xyxy, np.float32), np.array(out_s, np.float32), np.array(out_c)


def predict_tiled(detect, hflip=False, min_area=200, max_det=500):
    """detect(list of PIL tiles) -> list of (xyxy, scores, labels) in tile pixels."""
    parts = []
    for i in TEST_IDS:
        W, H = test_sizes[i]
        xs, ys = tile_origins(W), tile_origins(H)
        grid = [(a, b) for b in range(len(ys)) for a in range(len(xs))]
        with Image.open(TEST_IMAGES / f"{i}.jpg") as im:
            im = im.convert("RGB")
            tiles = [im.crop((xs[a], ys[b], xs[a] + TILE, ys[b] + TILE)) for a, b in grid]

        views = []
        for flipped in [False, True] if hflip else [False]:
            view = [t.transpose(Image.FLIP_LEFT_RIGHT) for t in tiles] if flipped else tiles
            kept = []
            for (a, b), (xyxy, s, c) in zip(grid, detect(view)):
                xyxy = xyxy.copy()
                if flipped:
                    xyxy[:, [0, 2]] = TILE - xyxy[:, [2, 0]]
                xyxy += [xs[a], ys[b], xs[a], ys[b]]
                xyxy[:, [0, 2]] = xyxy[:, [0, 2]].clip(0, W)
                xyxy[:, [1, 3]] = xyxy[:, [1, 3]].clip(0, H)
                (x_lo, x_hi), (y_lo, y_hi) = owned_span(xs, a, W), owned_span(ys, b, H)
                cx, cy = (xyxy[:, 0] + xyxy[:, 2]) / 2, (xyxy[:, 1] + xyxy[:, 3]) / 2
                area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
                keep = ((cx >= x_lo) & (cx < x_hi) & (cy >= y_lo) & (cy < y_hi)
                        & (area >= min_area) & (c < len(CLASSES)))
                kept.append((xyxy[keep], s[keep], c[keep]))
            views.append(nms(*(np.concatenate(v) for v in zip(*kept))))

        xyxy, s, c = fuse_flip_views(*views) if hflip else views[0]
        top = np.argsort(-s, kind="stable")[:max_det]
        parts.append(to_frame(i, xyxy[top], s[top], c[top]))
    return pd.concat(parts, ignore_index=True)


def yolo_detector(weights):
    model = YOLO(str(weights))

    def detect(tiles):
        out = []
        for k in range(0, len(tiles), 32):
            for r in model.predict(tiles[k:k + 32], imgsz=TILE, conf=0.001, iou=0.7, max_det=500,
                                   half=True, verbose=False):
                out.append((r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()))
        return out
    return detect


def rfdetr_detector(weights):
    model = RFDETRLarge(pretrain_weights=str(weights), resolution=TILE, num_classes=len(CLASSES))

    def detect(tiles):
        out = []
        for k in range(0, len(tiles), 16):
            dets = model.predict(tiles[k:k + 16], threshold=0.001)
            dets = dets if isinstance(dets, list) else [dets]  # a single image returns one Detections
            out += [(d.xyxy.astype(np.float32), d.confidence.astype(np.float32), d.class_id) for d in dets]
        return out
    return detect

In [ ]:
preds = {
    "rfdetr": predict_tiled(rfdetr_detector(WEIGHTS_RFDETR), hflip=True),
    "hakan": predict_full(WEIGHTS_BASE, max_det=300),
    "yolo26": predict_tiled(yolo_detector(WEIGHTS_YOLO26)),
    "ardahan": predict_full(WEIGHTS_RFS, max_det=1000),
}
for name, df in preds.items():
    df.to_csv(WORK / "preds" / f"{name}.csv", index=False)
    print(f"{name:8s} {len(df):>9,} boxes, {df.image_id.nunique():,} images")

## 9. Box fusion and submission

For each image and class, the four models' boxes are clustered and each cluster becomes one box:

1. Each box's confidence is multiplied by its model's weight: RF-DETR 1.0, YOLO11m 0.8, YOLO11m-rfs 0.6, YOLO26 0.4.
2. Going from the highest weighted score down, a box joins the cluster it overlaps most (IoU >= 0.55 with the
   cluster's current box), unless that cluster already holds a box from the same model. Otherwise it starts a new
   cluster.
3. The cluster's box is the score-weighted mean of its members. Its score is the best member's score, raised
   towards 1 when several models agree: `best + 0.3 * (n_models - 1) / 2 * (1 - best)`.
4. Each image keeps its 1000 highest-scoring boxes.

The weights and the 0.3 boost were chosen from a small grid on the scene_holdout_v2 val predictions. Each candidate
was scored with plain mAP@0.5 and with mAP reweighted to the test set's resolution x brightness mix.

In [ ]:
FUSION_WEIGHTS = {"rfdetr": 1.0, "hakan": 0.8, "yolo26": 0.4, "ardahan": 0.6}
FUSION_IOU, SUPPORT_BOOST, MIN_SCORE, MAX_DETS = 0.55, 0.3, 0.001, 1000


def iou_one_to_many(box, boxes):
    w = np.maximum(0.0, np.minimum(box[2], boxes[:, 2]) - np.maximum(box[0], boxes[:, 0]))
    h = np.maximum(0.0, np.minimum(box[3], boxes[:, 3]) - np.maximum(box[1], boxes[:, 1]))
    union = (box[2] - box[0]) * (box[3] - box[1]) + (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) - w * h
    return w * h / np.maximum(union, 1e-12)


def fuse_group(g):
    # float64: float32 can collapse thin boxes near x=1000 to zero width
    xyxy = np.column_stack([g.x, g.y, g.x + g.w, g.y + g.h]).astype(np.float64)
    scores, sources = g.score.to_numpy(np.float64), g.source.to_numpy()
    reps, members, member_scores, used = [], [], [], []
    for k in np.argsort(-scores, kind="stable"):
        match = -1
        eligible = np.array([sources[k] not in u for u in used], bool)
        if eligible.any():
            iou = iou_one_to_many(xyxy[k], np.vstack(reps))
            iou[~eligible] = -1
            if iou.max() >= FUSION_IOU:
                match = int(iou.argmax())
        if match < 0:
            reps.append(xyxy[k].copy())
            members.append([xyxy[k]])
            member_scores.append([scores[k]])
            used.append({sources[k]})
        else:
            members[match].append(xyxy[k])
            member_scores[match].append(scores[k])
            used[match].add(sources[k])
            reps[match] = np.average(members[match], axis=0, weights=member_scores[match])

    fused = []
    for box, ms, u in zip(reps, member_scores, used):
        if box[2] > box[0] and box[3] > box[1]:
            best = max(ms)
            conf = min(1.0, best + SUPPORT_BOOST * (len(u) - 1) / 2 * (1 - best))
            fused.append((conf, box[0], box[1], box[2] - box[0], box[3] - box[1]))
    return fused


combined = pd.concat([df.assign(source=name, score=df.conf * FUSION_WEIGHTS[name]) for name, df in preds.items()])
combined = combined[combined.score >= MIN_SCORE]

rows = [(image_id, label, *f)
        for (image_id, label), g in combined.groupby(["image_id", "label"], sort=False)
        for f in fuse_group(g)]
fused = pd.DataFrame(rows, columns=PRED_COLS).sort_values(["image_id", "conf"], ascending=[True, False])
fused = fused.groupby("image_id", sort=False).head(MAX_DETS)
fused.to_csv(WORK / "preds/fused.csv", index=False)

In [ ]:
def prediction_string(g):
    return " ".join(f"{r.label} {r.conf:.8g} {r.x:.8g} {r.y:.8g} {r.w:.8g} {r.h:.8g}" for r in g.itertuples())


strings = fused.groupby("image_id", sort=False).apply(prediction_string)
submission = sample.assign(PredictionString=sample.image_id.map(strings).fillna("none"))
submission.to_csv(WORK / "submission.csv", index=False)
print(f"{len(submission):,} images, {(submission.PredictionString == 'none').sum()} with no boxes")
submission.head()